# Кинетическая модель мембраны с двумя разными гранями — без постулата детального равновесия

**Цель.** Посчитать стационарное отношение $P_B/P_A$ для мембраны, у которой сторона B закрыта (поток $J=0$), только из кинетических уравнений с явными константами скоростей. Условие $\mu_H^A=\mu_H^B$ **не** закладывается, второе начало тоже.

**Схема (для каждой грани $X\in\{A,B\}$):**

$$\mathrm{H_2(gas)} \;\underset{k^X_{d}}{\overset{s^X Z P_X}{\rightleftarrows}}\; 2\,\mathrm{H(surf)} \;\underset{k^X_{bs}}{\overset{k^X_{sb}}{\rightleftarrows}}\; \mathrm{H(bulk)}_X \;\overset{D/l}{\longleftrightarrow}\; \mathrm{H(bulk)}_{\bar X}$$

Скорости (на единицу площади), $\Theta$ — покрытие, $c$ — концентрация в объёме под гранью, $N_s$ — плотность мест:

* адсорбция/десорбция: $r_1^X = 2\left[s^X Z P_X (1-\Theta_X)^2 - k_d^X \Theta_X^2\right]$ (атомов H);
* поверхность → объём: $r_2^X = k_{sb}^X\,\Theta_X - k_{bs}^X\,c_X(1-\Theta_X)$;
* диффузия: $J = \dfrac{D}{l}(c_A - c_B)$.

Константы у граней A и B **произвольные и независимые**.

In [ ]:
preparser(False)
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(1)

def rates(y, p):
    # y = [P_B, th_A, c_A, c_B, th_B];  P_A фиксировано (большой резервуар)
    PB, thA, cA, cB, thB = y
    r1A = 2*(p['sA']*p['Z']*p['PA']*(1-thA)**2 - p['kdA']*thA**2)
    r2A = p['ksbA']*thA - p['kbsA']*cA*(1-thA)
    J   = p['Dl']*(cA - cB)
    r2B = p['ksbB']*thB - p['kbsB']*cB*(1-thB)
    r1B = 2*(p['sB']*p['Z']*PB*(1-thB)**2 - p['kdB']*thB**2)
    return dict(r1A=r1A, r2A=r2A, J=J, r2B=r2B, r1B=r1B)

def rhs(y, p):
    r = rates(y, p)
    return np.array([
        -r['r1B']/(2*p['VB']),                 # газ в замкнутом объёме B
        (r['r1A'] - r['r2A'])/p['Ns'],
        (r['r2A'] - r['J'])/p['h'],
        (r['J'] + r['r2B'])/p['h'],
        (r['r1B'] - r['r2B'])/p['Ns']])

def integrate(y, p, t_end, n=4000):
    # неявный Эйлер с ньютоновскими итерациями и логарифмическим шагом (система жёсткая)
    ts = np.concatenate([[0], np.logspace(-6, np.log10(t_end), n)])
    Y = [y.copy()]
    for k in range(1, len(ts)):
        dt = ts[k]-ts[k-1]; z = Y[-1].copy()
        for _ in range(30):
            F = z - Y[-1] - dt*rhs(z, p)
            Jm = np.empty((5,5)); e = 1e-9
            for i in range(5):
                dz = np.zeros(5); dz[i] = e*max(1, abs(z[i]))
                Jm[:,i] = ((z+dz) - Y[-1] - dt*rhs(z+dz, p) - F)/dz[i]
            step = np.linalg.solve(Jm, -F); z = z + step
            if np.max(np.abs(step)/np.maximum(1e-12, np.abs(z))) < 1e-12: break
        Y.append(z)
    return ts, np.array(Y)
print('модель определена')

## 1. Теорема о закрытой цепочке

В стационаре $\dot y = 0$. Сторона B закрыта, значит $r_1^B=0$. Тогда из $\dot\Theta_B=0$ следует $r_2^B=0$, из $\dot c_B=0$ — $J=0$, из $\dot c_A=0$ — $r_2^A=0$, из $\dot\Theta_A=0$ — $r_1^A=0$.

Это **только сохранение числа атомов** в последовательной цепочке без ветвлений. Никакой термодинамики здесь нет. Каждое звено в стационаре имеет нулевую *нетто*-скорость. При этом $\Theta$ и $c$ на двух гранях могут быть любыми, совместимыми с уравнениями ниже.

Из нулевых нетто-скоростей:

$$\frac{\Theta_X}{1-\Theta_X} = \sqrt{K_1^X P_X},\quad K_1^X=\frac{s^X Z}{k_d^X};\qquad c_X = K_2^X\frac{\Theta_X}{1-\Theta_X},\quad K_2^X = \frac{k^X_{sb}}{k^X_{bs}};\qquad c_A=c_B .$$

Отсюда $c_X = S_X\sqrt{P_X}$, где **эффективная константа Сивертса грани** $S_X = K_2^X\sqrt{K_1^X}$, и

$$\boxed{\;\frac{P_B}{P_A} = \left(\frac{S_A}{S_B}\right)^2 = \frac{K_1^A\,(K_2^A)^2}{K_1^B\,(K_2^B)^2}\;}$$

Проверка ниже: решаем нестационарную систему со **случайными** константами и сравниваем предел $P_B(t\to\infty)$ с формулой. Заодно проверяем, что все нетто-скорости обнуляются.

In [ ]:
def random_params():
    lg = lambda a, b: 10**rng.uniform(a, b)
    return dict(Z=1.0, PA=lg(-1, 1), sA=lg(-2, 0), sB=lg(-2, 0),
                kdA=lg(-1, 1), kdB=lg(-1, 1), ksbA=lg(-1, 1), kbsA=lg(-1, 1),
                ksbB=lg(-1, 1), kbsB=lg(-1, 1), Dl=lg(-1, 1), Ns=1.0, h=1.0, VB=1.0)

def P_B_theory(p):
    S = lambda s, kd, ksb, kbs: (ksb/kbs)*np.sqrt(s*p['Z']/kd)
    SA = S(p['sA'], p['kdA'], p['ksbA'], p['kbsA']); SB = S(p['sB'], p['kdB'], p['ksbB'], p['kbsB'])
    return p['PA']*(SA/SB)**2

worst = 0; worst_r = 0
for trial in range(8):
    p = random_params()
    y0 = np.array([p['PA'], 0.0, 0.0, 0.0, 0.0])
    ts, Y = integrate(y0, p, 1e9, n=900)
    PBnum = Y[-1, 0]; PBth = P_B_theory(p)
    r = rates(Y[-1], p); rmax = max(abs(v) for v in r.values())
    worst = max(worst, abs(PBnum/PBth - 1)); worst_r = max(worst_r, rmax)
    print(f'{trial}: P_B/P_A числ = {PBnum/p["PA"]:.5g}, формула = {PBth/p["PA"]:.5g}')
print(f'max отн. ошибка P_B = {worst:.2e}; max |нетто-скорость| в конце = {worst_r:.2e} ->',
      'OK' if worst < 1e-6 and worst_r < 1e-8 else 'FAIL')

**Итог п. 1.** Кинетика без всяких постулатов даёт $P_B \neq P_A$, как только $S_A\neq S_B$. Это не вечный двигатель: $J=0$, поток не течёт, а давление на закрытой стороне просто устанавливается другим.

Весь вопрос сводится к одному числу — **отношению Вегшайдера для цикла**

$$W \equiv \frac{S_A^2}{S_B^2} = \frac{K_1^A (K_2^A)^2}{K_1^B (K_2^B)^2}.$$

Цикл: газ → грань A → объём → грань B → газ. Детальное равновесие (через то, что концентрация H в объёме — свойство металла, а не грани) требует $W=1$. Кинетика сама по себе $W$ не фиксирует.

## 2. Может ли такая мембрана дать работу?

Пусть $W\neq1$. Откроем сторону B через нагрузку (дроссель) в резервуар с $P_A$, получится замкнутый газовый контур при одной температуре. Если $W>1$, поток $J>0$ будет идти стационарно, и газ на дросселе можно заставить совершать работу $\approx J\,kT\ln(P_B/P_A)$.

Проверка ниже: открытая сторона B, $P_B=P_A$, стационарный $J$ по той же модели. Ожидание: $J\neq0$ ровно тогда, когда $W\neq1$, а знак $J$ совпадает со знаком $\ln W$.

In [ ]:
def steady_J_open(p):
    # P_B = P_A зафиксировано; ищем стационар по th_A, c_A, c_B, th_B методом Ньютона
    q = dict(p); 
    def F(z):
        y = np.array([q['PA'], *z]); return rhs(y, q)[1:]
    z = np.array([0.5, 1.0, 1.0, 0.5])
    for _ in range(200):
        f = F(z); Jm = np.empty((4,4))
        for i in range(4):
            dz = np.zeros(4); dz[i] = 1e-8*max(1, abs(z[i])); Jm[:,i] = (F(z+dz)-f)/dz[i]
        step = np.linalg.solve(Jm, -f); lam = 1.0
        while np.any((z+lam*step)[[0,3]] <= 0) or np.any((z+lam*step)[[0,3]] >= 1) or np.any((z+lam*step)[[1,2]] < 0): lam /= 2
        z = z + lam*step
        if np.max(np.abs(f)) < 1e-14: break
    return rates(np.array([q['PA'], *z]), q)['J']

ok = True
for trial in range(8):
    p = random_params()
    W = (P_B_theory(p)/p['PA'])
    J = steady_J_open(p)
    agree = (np.sign(J) == np.sign(np.log(W))) or abs(np.log(W)) < 1e-9
    ok &= agree
    print(f'{trial}: ln W = {np.log(W):+.3f}, J(P_B=P_A) = {J:+.4e}', '' if agree else '<- знак не совпал')
# симметричный случай W = 1: разные константы, но S_A = S_B
p = random_params(); 
p['ksbB'] = p['kbsB']*(p['ksbA']/p['kbsA'])*np.sqrt((p['sA']/p['kdA'])/(p['sB']/p['kdB']))
J1 = steady_J_open(p)
print(f'W = 1 при разных гранях: J = {J1:.2e} ->', 'OK' if abs(J1) < 1e-10 else 'FAIL')
print('знак J = знак ln W:', 'OK' if ok else 'FAIL')

## 3. Что о $W$ говорят данные Christmann

Изотермы Christmann дают **только $K_1$**: равновесие газ ⇄ поверхность на каждой грани отдельно. Это половина цикла. Константы поверхность ⇄ объём $K_2^X$ из них получить нельзя.

Поскольку реальные изотермы не ленгмюровские (есть латеральное взаимодействие), используем **локальную** $K_1$ при $\Theta=0.5$: тогда $\Theta/(1-\Theta)=1$ и $K_1^X = 1/P_X(\Theta=0.5)$.

Тогда $W=1$ эквивалентно требованию к подповерхностным константам:

$$\frac{K_2^{110}}{K_2^{111}} = \sqrt{\frac{K_1^{111}}{K_1^{110}}} = \sqrt{\frac{P_{110}(0.5)}{P_{111}(0.5)}} .$$

Это и есть **проверяемое предсказание постулата**. Если измеренное отношение $K_2$ (энергии подповерхностных мест, TDS-пики β/подповерхностного водорода на Pd(110) и Pd(111)) отличается от этого числа больше погрешности, $W\neq1$.

In [ ]:
TORR = 133.322368
def read_bln(f):
    rows = []
    for line in open(f):
        if '"' in line or not line.strip(): continue
        rows.append([float(x) for x in line.replace(' ', ',').split(',')[:3]])
    return np.array(rows)
DIR = 'didger'
def P_half(face, dphi):
    d = read_bln(f'{DIR}/Pd_{face}_p.bln'); out = {}
    for t in sorted(set(d[:,2])):
        m = d[:,2] == t; th = d[m,1]/dphi; lnP = np.log(d[m,0]*TORR); o = np.argsort(th)
        if th.min() <= 0.5 <= th.max(): out[t+273.15] = np.exp(np.interp(0.5, th[o], lnP[o]))
    return out
P110, P111 = P_half('110', 360.0), P_half('111', 180.0)
R = 8.314462618
print(' T, K   P110(0.5), Па  P111(0.5), Па   K1_111/K1_110   требуемое K2_110/K2_111')
Ts = sorted(set(P110) & set(P111))
for T in Ts:
    r = P110[T]/P111[T]
    print(f'{T:6.1f}   {P110[T]:.3e}     {P111[T]:.3e}     {r:8.3f}          {np.sqrt(r):.3f}')
# в энергетическом виде: ΔE_2 = E_2(110) - E_2(111), требуемая для W=1
lr = [0.5*np.log(P110[T]/P111[T]) for T in Ts]
b, a = np.polyfit(1/np.array(Ts), lr, 1)
print(f'\nln(K2_110/K2_111)_треб = {a:+.2f} {b:+.0f}/T  ->  ΔH_2(110−111) ≈ {-b*R/1000:+.1f} кДж/моль '
      f'(энтальпийная часть требования W=1)')

## 4. Выводы

**А. Получено без постулатов (проверено в ячейках 1–2)**

* В закрытой цепочке газ ⇄ поверхность ⇄ объём ⇄ поверхность ⇄ газ стационарное отношение $P_B/P_A = W$. Здесь $W$ — произведение констант равновесия звеньев по циклу. Это следует только из сохранения числа атомов.
* При открытой стороне и $P_A=P_B$ стационарный поток $J\neq0$ тогда и только тогда, когда $W\neq1$, а знак $J$ равен знаку $\ln W$. Значит, **мембрана с $W\neq1$ — это именно вечный двигатель 2-го рода** в смысле изотермического циклического контура. Кинетическая модель такое допускает: её уравнения согласованы при любом $W$.

**Б. Что добавляет детальное равновесие**

* Только одно утверждение: $W=1$ для любой пары граней. Термодинамически это значит, что $S_X$ — свойство объёма металла, а не грани. В кинетику это не встроено, это внешнее ограничение на константы.

**В. Как решить вопрос эмпирически**

* Изотермы Christmann фиксируют $K_1$ каждой грани (п. 3). Постулат $W=1$ при этом даёт **численное предсказание** для отношения подповерхностных констант $K_2^{110}/K_2^{111}$ и для соответствующей разности энтальпий.
* Проверка требует независимых измерений $K_2$ (подповерхностного/абсорбционного равновесия) на тех же гранях. Второй вариант — прямо измерить зависимость константы Сивертса от ориентации грани, через которую насыщался образец: $W = S_{110}^2/S_{111}^2$.
* Пока таких данных нет, ни «$W=1$», ни «$W\neq1$» из имеющихся файлов не следует.